In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_csv('data_intermediate/lead_service_line_addresses_geocoded.csv')
df.head()

,address,zip,town,address_full,suspected_lead,psl_materials,psl_other,service_line,csl_other,latitude,longitude
0,1 ALEXANDER AVENUE,7043,MONTCLAIR,"1 ALEXANDER AVENUE, MONTCLAIR, NJ, 07043",N,C,NaN,C,NaN,40.846594,-74.184035
1,1 ALEXANDER COURT,7043,MONTCLAIR,"1 ALEXANDER COURT, MONTCLAIR, NJ, 07043",N,C,NaN,C,NaN,40.849950,-74.195338
2,1 AMHERST PLACE,7043,MONTCLAIR,"1 AMHERST PLACE, MONTCLAIR, NJ, 07043",N,C,NaN,C,NaN,40.856101,-74.201105
3,1 ARGYLE ROAD,7043,MONTCLAIR,"1 ARGYLE ROAD, MONTCLAIR, NJ, 07043",Y,C,NaN,G,NaN,40.835567,-74.194835
4,1 BERKELEY PLACE,7042,MONTCLAIR,"1 BERKELEY PLACE, MONTCLAIR, NJ, 07042",N,C,NaN,C,NaN,40.828742,-74.214402


In [4]:
df_old=pd.read_csv('data_intermediate/lead_service_line_addresses_geocoded_old.csv')
df_old.head()

,address,zip,town,address_full,suspected_lead,psl_materials,psl_other,service_line,latitude,longitude
0,1 ALEXANDER AVENUE,7043,MONTCLAIR,"1 ALEXANDER AVENUE, MONTCLAIR, NJ, 07043",N,C,NaN,C,40.846734,-74.184007
1,1 ALEXANDER COURT,7043,MONTCLAIR,"1 ALEXANDER COURT, MONTCLAIR, NJ, 07043",N,C,NaN,C,40.849532,-74.195068
2,1 AMHERST PLACE,7043,MONTCLAIR,"1 AMHERST PLACE, MONTCLAIR, NJ, 07043",N,C,NaN,C,40.856096,-74.201631
3,1 ARGYLE ROAD,7043,MONTCLAIR,"1 ARGYLE ROAD, MONTCLAIR, NJ, 07043",Y,C,NaN,UX,40.835821,-74.195030
4,1 BERKELEY PLACE,7042,MONTCLAIR,"1 BERKELEY PLACE, MONTCLAIR, NJ, 07042",N,C,NaN,C,40.828901,-74.214466


In [8]:
print(df.shape)
print(df_old.shape)

(10593, 11)
(10591, 10)


In [18]:
l=0
for i in range(len(df)-2):
    if df.loc[i, 'service_line'] == df_old.loc[i, 'service_line']:
        l+=1

In [20]:
df[df['latitude'].isna()]

,address,zip,town,address_full,suspected_lead,psl_materials,psl_other,service_line,csl_other,latitude,longitude
43,1 HARVARD STREET,7042,MONTCLAIR,"1 HARVARD STREET, MONTCLAIR, NJ, 07042",Y,C,NaN,UX,NaN,NaN,NaN
46,1 KIPS RIDGE,7044,MONTCLAIR,"1 KIPS RIDGE, MONTCLAIR, NJ, 07044",N,C,NaN,C,NaN,NaN,NaN
51,1 LEWIS COURT,7042,MONTCLAIR,"1 LEWIS COURT, MONTCLAIR, NJ, 07042",N,C,NaN,C,NaN,NaN,NaN
75,1 STONEBRIDGE ROAD,7042,MONTCLAIR,"1 STONEBRIDGE ROAD, MONTCLAIR, NJ, 07042",Y,C,NaN,G,NaN,NaN,NaN
76,1 SUTHERLAND ROAD,7042,MONTCLAIR,"1 SUTHERLAND ROAD, MONTCLAIR, NJ, 07042",Y,C,NaN,L,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
10587,VALLEY ROAD,7043,MONTCLAIR,"VALLEY ROAD, MONTCLAIR, NJ, 07043",N,C,NaN,C,NaN,NaN,NaN
10589,WALNUT CRESCENT,7042,MONTCLAIR,"WALNUT CRESCENT, MONTCLAIR, NJ, 07042",N,C,NaN,C,NaN,NaN,NaN
10590,WALNUT STREET,7042,MONTCLAIR,"WALNUT STREET, MONTCLAIR, NJ, 07042",N,C,NaN,C,NaN,NaN,NaN
10591,WALNUT STREETATION,7042,MONTCLAIR,"WALNUT STREETATION, MONTCLAIR, NJ, 07042",N,C,NaN,C,NaN,NaN,NaN


In [21]:
for d in (df, df_old):
    d["match_key"] = d["address"].astype(str).str.upper().str.split().str.join(" ")
    d["latitude"] = pd.to_numeric(d["latitude"], errors="coerce")
    d["longitude"] = pd.to_numeric(d["longitude"], errors="coerce")

prior = (
    df_old.dropna(subset=["latitude", "longitude"])
          .drop_duplicates("match_key")[["match_key", "latitude", "longitude"]]
          .rename(columns={"latitude": "lat_old", "longitude": "lon_old"})
)

df = df.merge(prior, on="match_key", how="left")
df["coord_source"] = np.where(
    df["latitude"].notna(), "census",
    np.where(df["lat_old"].notna(), "google_prior", "none"))
df["latitude"] = df["latitude"].fillna(df["lat_old"])
df["longitude"] = df["longitude"].fillna(df["lon_old"])
df = df.drop(columns=["lat_old", "lon_old", "match_key"])

print(len(df), "rows |", df["latitude"].isna().sum(), "still missing")
print(df["coord_source"].value_counts())
placed = df.dropna(subset=["latitude", "longitude"])
print(placed[(placed.latitude < 40.78) | (placed.latitude > 40.90)
             | (placed.longitude < -74.26) | (placed.longitude > -74.15)]
      [["address", "latitude", "longitude", "coord_source"]])

10593 rows | 1 still missing
coord_source
census          10273
google_prior      319
none                1
Name: count, dtype: int64
                     address   latitude  longitude  coord_source
10229  9 STREETONEHENGE ROAD  39.825282 -74.202453  google_prior


In [23]:
df[df['latitude'].isna()]

,address,zip,town,address_full,suspected_lead,psl_materials,psl_other,service_line,csl_other,latitude,longitude,coord_source
10563,MIDLAND AVENUE-HENNINGBURG FIELD,7042,MONTCLAIR,"MIDLAND AVENUE-HENNINGBURG FIELD, MONTCLAIR, N...",N,C,NaN,C,NaN,NaN,NaN,none


In [24]:
MASK = df["address"] == "MIDLAND AVENUE-HENNINGBURG FIELD"
df.loc[MASK, ["latitude", "longitude"]] = [40.8239678526292, -74.2136549165529]
df.loc[MASK, "coord_source"] = "manual_google_maps"

print(df["latitude"].isna().sum(), "still missing")
print(df["coord_source"].value_counts())

0 still missing
coord_source
census                10273
google_prior            319
manual_google_maps        1
Name: count, dtype: int64


In [26]:
df.to_csv('data_intermediate/lead_service_line_addresses_geocoded.csv', index=False)

In [1]:
# Check
import pandas as pd
df=pd.read_csv('data_intermediate/lead_service_line_addresses_geocoded.csv')

In [7]:
df.isna().sum()

address               0
zip                   0
town                  0
address_full          0
suspected_lead        0
psl_materials         0
psl_other         10589
service_line          1
csl_other         10482
latitude              0
longitude             0
coord_source          0
dtype: int64

In [5]:
import numpy as np

ann = df.copy()
ann["flag_incomplete"] = np.where(ann["address"].str.match(r"^\d", na=False), 0, 1)
ann["latitude_adj"] = ann["latitude"]
ann["longitude_adj"] = ann["longitude"]
ann["address_full_adj"] = ann["address_full"]
ann.loc[ann["address"] == "MIDLAND AVENUE-HENNINGBURG FIELD", "flag_incomplete"] = 0

ann.to_csv("data/lead_service_line_addresses_geocoded_annotated.csv", index=False)
print(len(ann), (ann["flag_incomplete"] == 1).sum(), ann["latitude_adj"].isna().sum())

10593 55 0


In [4]:
import os
print(os.path.abspath("data/lead_service_line_addresses_geocoded_annotated.csv"))

c:\Users\Atharv\Projects\montclairlocal\lead-service-line-etl_August2026\data\lead_service_line_addresses_geocoded_annotated.csv


In [6]:
import geopandas as gpd
g = ann.dropna(subset=["latitude_adj", "longitude_adj"])
gpd.GeoDataFrame(g, geometry=gpd.points_from_xy(g.longitude_adj, g.latitude_adj),
                 crs="EPSG:4326").to_file(
    "data/lead_service_line_addresses_geocoded_annotated.gpkg", driver="GPKG")

In [7]:
fixes = {
    "9 STREETONEHENGE ROAD": (40.847608, -74.188673),
    "10 OVERLOOK PARK":      (40.848581, -74.194198),
}
for addr, (lat, lon) in fixes.items():
    m = df["address"] == addr
    print(addr, "->", m.sum(), "rows")
    df.loc[m, ["latitude", "longitude"]] = [lat, lon]
    df.loc[m, "coord_source"] = "manual_google_maps"

9 STREETONEHENGE ROAD -> 1 rows
10 OVERLOOK PARK -> 1 rows


In [8]:
df.to_csv("data_intermediate/lead_service_line_addresses_geocoded.csv", index=False)

In [9]:
import numpy as np
import geopandas as gpd

ann = df.copy()
ann["flag_incomplete"] = np.where(ann["address"].str.match(r"^\d", na=False), 0, 1)
ann["latitude_adj"] = ann["latitude"]
ann["longitude_adj"] = ann["longitude"]
ann["address_full_adj"] = ann["address_full"]
ann.loc[ann["address"] == "MIDLAND AVENUE-HENNINGBURG FIELD", "flag_incomplete"] = 0

ann.to_csv("data/lead_service_line_addresses_geocoded_annotated.csv", index=False)

g = ann[ann["flag_incomplete"] == 0].dropna(subset=["latitude_adj", "longitude_adj"])
gpd.GeoDataFrame(
    g, geometry=gpd.points_from_xy(g.longitude_adj, g.latitude_adj), crs="EPSG:4326"
).to_file("data/lead_service_line_addresses_geocoded_annotated.gpkg", driver="GPKG")

print(len(ann), "rows |", len(g), "features")

10593 rows | 10538 features
